# Partie 1 : mise en place du cluster et positionnement

**Objectif :** vérifier que l'environnement Docker fonctionne (Spark Master + 2 Workers, MinIO, Kafka) et
afficher les informations de la `SparkSession` demandées par le cahier des charges.

Avant d'exécuter ce notebook : `docker compose up -d --build` (voir le README).

In [1]:
from sahlpay_spark import get_spark, list_lake, LAKE, MASTER

spark = get_spark("SahlPay-Partie1-Cluster")
sc = spark.sparkContext

print("Version de Spark      :", spark.version)
print("Master                :", sc.master)
print("appName               :", sc.appName)
print("defaultParallelism    :", sc.defaultParallelism, "(nombre de coeurs disponibles)")
print("Application ID        :", sc.applicationId)
print("Spark UI (Driver)     :", sc.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 21:00:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version de Spark      : 3.5.3
Master                : spark://spark-master:7077
appName               : SahlPay-Partie1-Cluster
defaultParallelism    : 4 (nombre de coeurs disponibles)
Application ID        : app-20261125210031-0003
Spark UI (Driver)     : http://jupyter:4040


## Executors enregistrés

`getExecutorMemoryStatus` liste le Driver et chaque Executor (un par Worker ici).

In [2]:
status = sc._jsc.sc().getExecutorMemoryStatus()
it = status.keySet().iterator()
noeuds = []
while it.hasNext():
    noeuds.append(it.next())
print("Noeuds (Driver + Executors) :", noeuds)
print("Nombre d'Executors :", len(noeuds) - 1)

# Petit job distribué : chaque partition renvoie le nom de la machine qui l'a traitée
import socket
hotes = (sc.parallelize(range(sc.defaultParallelism * 4), sc.defaultParallelism * 2)
           .map(lambda _: socket.gethostname())
           .distinct()
           .collect())
print("Tâches exécutées sur :", hotes)

Noeuds (Driver + Executors) : ['jupyter:37567', '172.28.0.6:46521', '172.28.0.3:46711']
Nombre d'Executors : 2


[Stage 0:>                                                          (0 + 4) / 8]

Tâches exécutées sur : ['spark-worker-1', 'spark-worker-2']


## Vérification du lac MinIO (bucket `lake`)

On écrit puis relit un petit fichier dans le bucket, et on affiche l'arborescence du lac.

In [3]:
test = spark.createDataFrame([("ok", 1)], ["statut", "valeur"])
test.write.mode("overwrite").json(f"{LAKE}/checkpoints/_test_partie1")
print("Relecture :", spark.read.json(f"{LAKE}/checkpoints/_test_partie1").collect())
list_lake(spark, depth=2)

26/11/25 21:00:42 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/11/25 21:00:44 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/25 21:00:48 WARN DataSource: All paths were ignored:                      
  s3a://lake/checkpoints/_test_partie1


Relecture : [Row(statut='ok', valeur=1)]
s3a://lake
  bronze/
    history/
  checkpoints/
    _test_partie1/
  reference/
    merchants.json  (44504 octets)


## Vérification du topic Kafka `transactions`

In [4]:
import os
try:
    from kafka import KafkaConsumer
    consumer = KafkaConsumer(bootstrap_servers=os.environ.get("KAFKA_BOOTSTRAP", "kafka:9092"))
    partitions = consumer.partitions_for_topic("transactions")
    print("Topic transactions, partitions :", sorted(partitions) if partitions else "topic introuvable")
    consumer.close()
except Exception as e:
    print("Kafka non joignable depuis ce notebook :", e)

Topic transactions, partitions : [0, 1, 2]


## Captures à faire pour le rapport

1. Spark Master UI : <http://localhost:8080> avec les **2 Workers ALIVE** et l'application `SahlPay-Partie1-Cluster`.
2. Console MinIO : <http://localhost:9001> avec le bucket `lake`.
3. La sortie de la première cellule (version, master, appName, defaultParallelism).

In [5]:
spark.stop()